<a href="https://colab.research.google.com/github/VanRaWin/GSLC_Sesi5/blob/main/GSLC_Sesi5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import numpy as np
import random
import time
import pandas as pd
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan Device: {device}")


transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


print("Memuat dataset...")
full_trainval = datasets.OxfordIIITPet(root='./data', split='trainval', download=True, transform=transform_pipeline)
full_test = datasets.OxfordIIITPet(root='./data', split='test', download=True, transform=transform_pipeline)


selected_classes = list(range(10))

def get_fast_indices(dataset, max_samples=150):
    indices = []

    for idx in range(len(dataset)):
        _, label = dataset[idx]
        if label in selected_classes:
            indices.append(idx)
        if len(indices) >= max_samples:
            break
    return indices

train_idx = get_fast_indices(full_trainval, max_samples=140)
val_idx = get_fast_indices(full_trainval, max_samples=40)
test_idx = get_fast_indices(full_test, max_samples=50)

train_loader = DataLoader(Subset(full_trainval, train_idx), batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(full_trainval, val_idx), batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(Subset(full_test, test_idx), batch_size=16, shuffle=False, num_workers=0)

print(f"Subset Data Siap -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")


def run_fast_experiment(condition_code):
    print(f"\n>>> Mengeksekusi Kondisi: {condition_code} ...")
    start_time = time.time()


    if condition_code == 'B1':
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 10)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    elif condition_code == 'B2':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, 10)
        optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
    elif condition_code in ['B3', 'B4-1']:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        for param in model.layer4.parameters(): param.requires_grad = True
        model.fc = nn.Linear(model.fc.in_features, 10)
        optimizer = torch.optim.AdamW([
            {"params": model.layer4.parameters(), "lr": 1e-4},
            {"params": model.fc.parameters(), "lr": 1e-3}
        ])
    elif condition_code == 'B5':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for param in model.parameters(): param.requires_grad = False
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 10)
        optimizer = torch.optim.AdamW(model.classifier[1].parameters(), lr=1e-3)


    model = model.to(device)
    criterion = nn.CrossEntropyLoss()


    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()


    model.eval()
    val_correct, total_val = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            _, preds = torch.max(model(images), 1)
            val_correct += (preds == labels).sum().item()
            total_val += labels.size(0)
    val_acc = val_correct / total_val if total_val > 0 else 0.0


    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            _, preds = torch.max(model(images), 1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

    test_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    duration = time.time() - start_time

    print(f"Hasil {condition_code} -> Val Acc: {val_acc:.4f} | Test Macro F1: {test_f1:.4f}")

    return {
        "Kondisi": condition_code,
        "Epoch": 1,
        "Waktu (s)": round(duration, 2),
        "Akurasi Validasi": round(val_acc, 4),
        "Macro F1-Score": round(test_f1, 4)
    }


history = []
for k in ['B1', 'B2', 'B3', 'B4-1', 'B5']:
    res = run_fast_experiment(k)
    history.append(res)


df_final = pd.DataFrame(history)
df_final.to_csv("log_hasil_mentah.csv", index=False)
print("\n=== EKSPERIMEN SELESAI, TABEL DI BAWAH INI ADALAH HASIL ANDA ===")
print(df_final)


Menggunakan Device: cpu
Memuat dataset...
Subset Data Siap -> Train: 140 | Val: 40 | Test: 50

>>> Mengeksekusi Kondisi: B1 ...
Hasil B1 -> Val Acc: 0.0000 | Test Macro F1: 0.0000

>>> Mengeksekusi Kondisi: B2 ...
Hasil B2 -> Val Acc: 0.9250 | Test Macro F1: 0.3158

>>> Mengeksekusi Kondisi: B3 ...
Hasil B3 -> Val Acc: 1.0000 | Test Macro F1: 0.3265

>>> Mengeksekusi Kondisi: B4-1 ...
Hasil B4-1 -> Val Acc: 1.0000 | Test Macro F1: 0.3194

>>> Mengeksekusi Kondisi: B5 ...
Hasil B5 -> Val Acc: 1.0000 | Test Macro F1: 0.3265

=== EKSPERIMEN SELESAI, TABEL DI BAWAH INI ADALAH HASIL ANDA ===
  Kondisi  Epoch  Waktu (s)  Akurasi Validasi  Macro F1-Score
0      B1      1      48.14             0.000          0.0000
1      B2      1      22.66             0.925          0.3158
2      B3      1      29.46             1.000          0.3265
3    B4-1      1      30.76             1.000          0.3194
4      B5      1      17.48             1.000          0.3265
